# Projet 14 : Génération de poèmes avec un modèle plus grand (Qwen2.5-1.5B + LoRA)

**Objectif :** obtenir des poèmes avec plus de sens que DistilGPT-2, en adaptant un modèle de 1,5 milliard de paramètres (déjà bon en langage) à vos poèmes, avec la méthode **LoRA** (on n'entraîne qu'environ 1 % des poids).

**Ce que fait ce notebook, dans l'ordre**

| # | Partie | En une phrase |
|---|---|---|
| 1 | Vérifier le GPU | Le GPU T4 doit être actif |
| 2 | Installer les bibliothèques | `transformers`, `peft`, `accelerate` |
| 3 | Réglages + Google Drive | Chemins, mode test, hyperparamètres |
| 4 | Charger les données | CSV du dossier Drive, nettoyage |
| 5 | Préparer les exemples | Thème + première ligne → poème |
| 6 | Charger le modèle + LoRA | Modèle de base + petits adaptateurs |
| 7 | Entraîner | Avec sauvegarde et reprise automatique |
| 8 | Évaluer | Perte, perplexité par token et par mot |
| 9 | Générer 10 poèmes | Mêmes débuts de poèmes que précédemment |
| 10 | Essayer vos propres poèmes | Vous donnez un thème et une première ligne |
| 11 | Sauvegarder | L'adaptateur LoRA sur Drive + zip |

**Avant de commencer**
1. *Exécution → Modifier le type d'exécution → **T4 GPU*** .
2. Mettez dans votre Google Drive le dossier `projet_poesie` avec, au choix :
   - `data/processed/train.csv`, `val.csv`, `test.csv` (le découpage de votre projet local), **ou**
   - le fichier brut `PoetryFoundationData.csv` (le notebook fait alors le nettoyage et le découpage 80/10/10 lui-même).
3. **Première exécution : laissez `TEST_MODE = True`** (20 pas, quelques minutes) pour vérifier qu'il n'y a pas d'erreur. Ensuite, passez `TEST_MODE = False` et relancez tout (*Exécution → Tout exécuter*).

## 1. Vérifier le GPU

**En clair :** un modèle de 1,5 milliard de paramètres ne s'entraîne pas sur un processeur normal. Il faut le GPU de Colab. Cette cellule s'arrête avec un message si le GPU n'est pas actif.

In [15]:
import torch

print("GPU disponible :", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("PAS DE GPU : menu Exécution > Modifier le type d'exécution > T4 GPU, puis relancer.")
print("Carte :", torch.cuda.get_device_name(0))
print("Mémoire GPU : %.1f Go" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

GPU disponible : True
Carte : Tesla T4
Mémoire GPU : 15.6 Go


## 2. Installer les bibliothèques

**En clair :** `transformers` charge le modèle, `peft` ajoute les adaptateurs LoRA, `accelerate` gère le calcul sur GPU.
Si Colab affiche « Redémarrer la session », faites-le, puis relancez le notebook depuis le début.

In [16]:
!pip install -q -U transformers peft accelerate

In [1]:
# Mise à jour de torchao pour corriger le conflit de compatibilité avec PEFT
!pip install -q -U torchao

In [2]:
!pip -q install kagglehub
%pip install --upgrade --force-reinstall --no-cache-dir pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 196.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 259.1 MB/s eta 0:00:00
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.6
    Uninstalling pandas-3.0.6:
      Successfully uninstalled pandas-3.0.6
ERROR: pip's dependency resolv

## 3. Réglages et Google Drive

**En clair :** on connecte Drive (autorisez l'accès quand Colab le demande) et on règle tout au même endroit.

- `TEST_MODE = True` : test rapide de 20 pas, pour détecter les erreurs.
- `TEST_MODE = False` : entraînement complet.
- `BASE` : le dossier de votre projet dans Drive. Modifiez-le si votre dossier porte un autre nom.

In [1]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "tgdivy/poetry-foundation-poems"
)

print("Chemin du dataset :", dataset_path)

Using Colab cache for faster access to the 'poetry-foundation-poems' dataset.
Chemin du dataset : /kaggle/input/poetry-foundation-poems


In [2]:
import pandas as pd
import glob
import os

# Recherche des fichiers CSV dans le dossier téléchargé
csv_files = glob.glob(os.path.join(dataset_path, "**/*.csv"), recursive=True)

if csv_files:
    df = pd.read_csv(csv_files[0])
    print("Dimensions :", df.shape)
    print("Colonnes :", df.columns.tolist())
    display(df.head())
else:
    print("Aucun fichier CSV trouvé.")

Dimensions : (13854, 5)
Colonnes : ['Unnamed: 0', 'Title', 'Poem', 'Poet', 'Tags']


,Unnamed: 0,Title,Poem,Poet,Tags
0,0,\r\r\n Objects Used to Prop...,"\r\r\nDog bone, stapler,\r\r\ncribbage board, ...",Michelle Menting,NaN
1,1,\r\r\n The New Church\r\r\n...,"\r\r\nThe old cupola glinted above the clouds,...",Lucia Cherciu,NaN
2,2,\r\r\n Look for Me\r\r\n ...,\r\r\nLook for me under the hood\r\r\nof that ...,Ted Kooser,NaN
3,3,\r\r\n Wild Life\r\r\n ...,"\r\r\nBehind the silo, the Mother Rabbit\r\r\n...",Grace Cavalieri,NaN
4,4,\r\r\n Umbrella\r\r\n ...,\r\r\nWhen I push your button\r\r\nyou fly off...,Connie Wanek,NaN


In [3]:
# Vérifier les valeurs manquantes
print("Valeurs manquantes :")
print(df.isnull().sum())

# Vérifier les poèmes vides
print("\nPoèmes vides :")
print(df["Poem"].fillna("").str.strip().eq("").sum())

# Calculer la longueur des poèmes
longueurs = df["Poem"].dropna().str.len()

print("\nLongueur moyenne :", longueurs.mean())
print("Longueur maximale :", longueurs.max())
print("Longueur minimale :", longueurs.min())

Valeurs manquantes :
Unnamed: 0      0
Title           0
Poem            0
Poet            0
Tags          955
dtype: int64

Poèmes vides :
101

Longueur moyenne : 1505.4722823733218
Longueur maximale : 88699
Longueur minimale : 4


In [4]:
from google.colab import drive
drive.mount("/content/drive")

import os, glob, re, math, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ============ RÉGLAGES ============

TEST_MODE = True

# Dataset déjà téléchargé depuis Kaggle
DATA_DIR = dataset_path

# Drive uniquement pour les sauvegardes
BASE = "/content/drive/MyDrive/projet_poesie"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_LEN = 320
MIN_LINES = 4
MAX_LINES = 16

EPOCHS = 1
LR = 2e-4
BATCH = 4
ACCUM = 4
SEED = 42

# ==================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RUN_DIR = f"{BASE}/lora_runs/{'test' if TEST_MODE else 'full'}"
OUT_DIR = f"{BASE}/outputs"

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print("Mode :", "TEST RAPIDE" if TEST_MODE else "ENTRAÎNEMENT COMPLET")
print("Dataset Kaggle :", DATA_DIR)
print("Dossier de sauvegarde :", BASE)
print("GPU disponible :", torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mode : TEST RAPIDE
Dataset Kaggle : /kaggle/input/poetry-foundation-poems
Dossier de sauvegarde : /content/drive/MyDrive/projet_poesie
GPU disponible : True


## 4. Charger et préparer les données

**En clair :** on lit les poèmes, on les nettoie (guillemets, espaces) en **gardant les sauts de ligne** (les vers et les strophes), et on garde seulement les poèmes de 4 à 16 vers. Un poème court garde mieux son sens qu'un poème long.

Pour chaque poème, on prépare aussi :
- un **thème** (les mots-clés `Tags` du dataset, ou le titre s'il n'y en a pas) ;
- la **première ligne**.

Le modèle apprendra donc : *thème + première ligne → poème complet*.

In [5]:
TRANS = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"', "—": "-", "–": "-", "\u00a0": " "})


def clean(t):
    t = str(t).replace("\r", "").translate(TRANS)
    t = re.sub(r"[ \t]+", " ", t)                      # espaces multiples
    t = "\n".join(l.strip() for l in t.split("\n"))    # espaces en bord de ligne
    t = re.sub(r"\n{3,}", "\n\n", t)                   # strophe = double saut de ligne
    return t.strip()


def get_col(df, name):
    for c in df.columns:
        if str(c).strip().lower() == name:
            return c
    return None


def make_theme(tags, title):
    if tags is not None and pd.notna(tags) and str(tags).strip():
        parts = [p.strip() for p in str(tags).split(",") if p.strip()][:4]
        if parts:
            return ", ".join(parts)
    if title is not None and pd.notna(title) and str(title).strip():
        return str(title).strip()
    return "life"


def to_rows(df):
    pc, tc, ti = get_col(df, "poem"), get_col(df, "tags"), get_col(df, "title")
    if pc is None:
        raise KeyError(f"Colonne 'Poem' introuvable. Colonnes présentes : {list(df.columns)}")
    rows = []
    for _, r in df.iterrows():
        if pd.isna(r[pc]):
            continue
        poem = clean(r[pc])
        lines = [l for l in poem.split("\n") if l.strip()]
        if not (MIN_LINES <= len(lines) <= MAX_LINES):
            continue
        rows.append({
            "theme": make_theme(r[tc] if tc else None, r[ti] if ti else None),
            "first": lines[0].strip(),
            "poem": poem,
            "n_words": len(poem.split()),
        })
    return rows

In [6]:
# ============================================================
# CHARGEMENT ET PRÉPARATION DES DONNÉES DE POÉSIE
# Source : Kaggle
# Objectif : préparer les données pour Qwen + LoRA
# ============================================================

# 1. Récupérer le dataset déjà chargé depuis Kaggle
raw = df.copy()

# 2. Identifier la colonne contenant les poèmes
pc = get_col(raw, "poem")

if pc is None:
    raise KeyError(
        f"Colonne 'Poem' introuvable. "
        f"Colonnes disponibles : {list(raw.columns)}"
    )

print("Nombre initial de poèmes :", len(raw))

# 3. Supprimer les valeurs manquantes
raw = raw.dropna(subset=[pc]).copy()

# 4. Nettoyer les textes avec la fonction clean()
raw[pc] = raw[pc].map(clean)

# 5. Supprimer les poèmes vides
raw = raw[raw[pc].str.strip() != ""].copy()

# 6. Supprimer les poèmes en double
raw = raw.drop_duplicates(subset=[pc]).copy()

# 7. Garder les poèmes de moins de 2000 caractères
# et contenant au moins 4 vers non vides
raw = raw[
    (raw[pc].str.len() < 2000) &
    (
        raw[pc].map(
            lambda p: sum(
                bool(line.strip())
                for line in p.split("\n")
            )
        ) >= MIN_LINES
    )
].copy()

print("Poèmes après nettoyage :", len(raw))

# 8. Mélanger les données pour éviter un ordre particulier
raw = raw.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

# 9. Diviser le dataset : 80 % / 10 % / 10 %
n = len(raw)

train_df = raw.iloc[:int(0.8 * n)].copy()

val_df = raw.iloc[
    int(0.8 * n):int(0.9 * n)
].copy()

test_df = raw.iloc[
    int(0.9 * n):
].copy()

print("\n--- Répartition des données ---")
print("Train :", len(train_df))
print("Validation :", len(val_df))
print("Test :", len(test_df))

# 10. Transformer les données en exemples pour Qwen
# On utilise la fonction to_rows() de la première cellule

train_rows = to_rows(train_df)
val_rows = to_rows(val_df)
test_rows = to_rows(test_df)

print("\n--- Poèmes retenus (4 à 16 vers) ---")
print("Train :", len(train_rows))
print("Validation :", len(val_rows))
print("Test :", len(test_rows))

# 11. Vérifier la présence d'exemples
if not train_rows:
    raise ValueError(
        "Aucun poème disponible après préparation."
    )

# 12. Afficher un exemple
print("\n========== EXEMPLE ==========")

ex = train_rows[0]

print("Thème :", ex["theme"])
print("Première ligne :", ex["first"])
print("Nombre de mots :", ex["n_words"])

print("\nPoème :")
print(ex["poem"][:400])

print("\nPréparation terminée !")

Nombre initial de poèmes : 13854
Poèmes après nettoyage : 7274

--- Répartition des données ---
Train : 5819
Validation : 727
Test : 728

--- Poèmes retenus (4 à 16 vers) ---
Train : 1962
Validation : 255
Test : 264

========== EXEMPLE ==========
Thème : Activities, Sports & Outdoor Activities
Première ligne : if this is a game then we have made it, unknowing,
Nombre de mots : 104

Poème :
if this is a game then we have made it, unknowing,
to the final four. unlikely underdogs. spectators turned
to suspect sport. anti-athletes. out of shape beyond reason.
at season's height we fight for a limited audience. few dancers.
fewer cheers. down by 30 and our coach m.i.a. we, foolish, dribble.
each bounce-back brings a stranger. can't call us for traveling because
we ain't going nowhere. in

Préparation terminée !


## 5. Préparer les exemples d'entraînement

**En clair :** on écrit chaque exemple comme une petite conversation :

- *Question :* « Écris un poème sur : {thème}. Commence par cette ligne : {première ligne} »
- *Réponse :* le poème complet.

Le modèle n'est corrigé **que sur la réponse** (le poème), pas sur la question : dans les données, les positions de la question reçoivent l'étiquette `-100`, que l'entraînement ignore.

In [7]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("Fin de réponse (eos) :", tok.eos_token)

SYSTEM = "You are a poet. Write original English poems with clear imagery and a coherent theme."


def prompt_text(theme, first):
    user = f"Write a poem about: {theme}.\nStart with this line: {first}"
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def encode(row):
    p = tok(prompt_text(row["theme"], row["first"]), add_special_tokens=False)["input_ids"]
    a = tok(row["poem"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    if len(p) >= MAX_LEN - 8:
        return None
    ids = (p + a)[:MAX_LEN]
    lab = ([-100] * len(p) + a)[:MAX_LEN]
    return {"input_ids": ids, "labels": lab, "n_words": row["n_words"]}


def encode_all(rows):
    out = [encode(r) for r in rows]
    return [o for o in out if o is not None]


train_ds, val_ds, test_ds = encode_all(train_rows), encode_all(val_rows), encode_all(test_rows)
print(f"Exemples encodés : train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}")
print("Longueur moyenne (tokens) : %.0f" % np.mean([len(x["input_ids"]) for x in train_ds]))


def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), m), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(batch), m), dtype=torch.long)
    lab = torch.full((len(batch), m), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        att[i, :n] = 1
        lab[i, :n] = torch.tensor(b["labels"])
    return {"input_ids": ids, "attention_mask": att, "labels": lab}

Fin de réponse (eos) : <|im_end|>
Exemples encodés : train=1962 | val=255 | test=264
Longueur moyenne (tokens) : 189


In [8]:
# Vérifier les données tokenisées

print("Train :", len(train_ds))
print("Validation :", len(val_ds))
print("Test :", len(test_ds))

# Vérifier la longueur des séquences
longueurs = [len(x["input_ids"]) for x in train_ds]

if longueurs:
    print("Longueur moyenne :", np.mean(longueurs))
    print("Longueur maximale :", max(longueurs))
    print("Longueur minimale :", min(longueurs))

    # Vérifier combien d'exemples atteignent MAX_LEN
    limites = sum(l == MAX_LEN for l in longueurs)
    print("Exemples atteignant MAX_LEN :", limites)
else:
    print("Attention : aucun exemple tokenisé.")

Train : 1962
Validation : 255
Test : 264
Longueur moyenne : 188.55300713557594
Longueur maximale : 320
Longueur minimale : 67
Exemples atteignant MAX_LEN : 111


## 6. Charger le modèle et ajouter LoRA

**En clair :** on charge **Qwen2.5-1.5B-Instruct** (un modèle déjà bon en anglais) et on ajoute de petits adaptateurs **LoRA** dans ses couches. Seuls ces adaptateurs sont entraînés : c'est rapide et ça tient dans la mémoire du GPU gratuit.

Le modèle est chargé en précision demi (`float16`), adaptée au GPU T4. Les adaptateurs, eux, sont entraînés en `float32` pour que l'entraînement reste stable.

In [9]:
# ====================================================
# CONFIGURATION DU MODÈLE QWEN + LORA
# ====================================================

import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# 1. Vérifier le GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU non disponible. Active le GPU T4 dans Colab."
    )

print("GPU utilisé :", torch.cuda.get_device_name(0))

# 2. Charger Qwen (en utilisant le nouveau paramètre dtype pour éviter le warning)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16
).to("cuda")

# 3. Désactiver le cache pendant l'entraînement
model.config.use_cache = False

# 4. Réduire la consommation de mémoire
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

model.enable_input_require_grads()

# 5. Configuration LoRA
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# 6. Ajouter les adaptateurs LoRA
model = get_peft_model(model, lora_cfg)

# 7. Garder les paramètres entraînables en float32
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

# 8. Afficher les paramètres entraînables
print("\n--- Paramètres LoRA ---")
model.print_trainable_parameters()

print("\nConfiguration Qwen + LoRA terminée !")

GPU utilisé : Tesla T4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


--- Paramètres LoRA ---
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

Configuration Qwen + LoRA terminée !


## 7. Entraîner

**En clair :** le modèle lit les poèmes par lots de 16 et corrige ses adaptateurs pour mieux prédire le mot suivant : on minimise la perte

L = −(1/N) · Σ log P(mot_t | mots précédents)

**Sauvegarde et reprise :** un checkpoint est enregistré dans Drive tous les 100 pas. Si Colab se coupe, relancez les cellules 1 à 6 puis celle-ci : l'entraînement reprend au dernier checkpoint.

**Durée :**
- Pas par époque ≈ nombre d'exemples d'entraînement / 16.
- Durée ≈ nombre de pas × secondes par pas.
- Après le test rapide, la cellule affiche une **estimation du temps de l'entraînement complet** (à titre indicatif, de l'ordre de 30 à 60 minutes).

Gardez l'onglet Colab ouvert pendant l'entraînement.

In [ ]:
# ============================================================
# ENTRAÎNEMENT QWEN + LORA
# Projet NLP : Génération automatique de poésie
# Google Colab - GPU T4
# ============================================================

import os
import math
import time
import torch

from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint


# ============================================================
# 1. CRÉATION DU DATASET PYTORCH
# ============================================================

class PoemDS(torch.utils.data.Dataset):

    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


# ============================================================
# 2. VÉRIFICATION DES DONNÉES ET DU GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU indisponible. Active le GPU T4 dans Colab."
    )

if len(train_ds) == 0:
    raise ValueError(
        "Le dataset d'entraînement est vide."
    )

print("GPU :", torch.cuda.get_device_name(0))

print("Exemples entraînement :", len(train_ds))
print("Exemples validation :", len(val_ds))
print("Exemples test :", len(test_ds))


# ============================================================
# 3. CONFIGURATION DES DOSSIERS DE SAUVEGARDE
# ============================================================

# Séparer les entraînements TEST et COMPLET

MODE_NAME = "test" if TEST_MODE else "full"

RUN_DIR = os.path.join(
    BASE,
    "lora_runs",
    MODE_NAME
)

ADAPTER_DIR = os.path.join(
    BASE,
    "outputs",
    f"adapter_lora_{MODE_NAME}"
)

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print("\nMode :", MODE_NAME)
print("Checkpoints :", RUN_DIR)
print("Adaptateur :", ADAPTER_DIR)


# ============================================================
# 4. CALCUL DES ÉTAPES
# ============================================================

steps_full = math.ceil(
    len(train_ds) / (BATCH * ACCUM)
) * EPOCHS

print("\nÉtapes estimées pour l'entraînement complet :", steps_full)

if TEST_MODE:
    print("TEST RAPIDE : 20 étapes")
else:
    print("ENTRAÎNEMENT COMPLET")


# ============================================================
# 5. CONFIGURATION TRAINING ARGUMENTS
# ============================================================

args = TrainingArguments(

    # Dossier de sauvegarde
    output_dir=RUN_DIR,

    # Nombre d'époques
    num_train_epochs=EPOCHS,

    # 20 étapes seulement pour le test
    max_steps=20 if TEST_MODE else -1,

    # Batch size
    per_device_train_batch_size=BATCH,

    # Accumulation des gradients
    gradient_accumulation_steps=ACCUM,

    # Learning rate
    learning_rate=LR,

    # Scheduler
    lr_scheduler_type="cosine",

    # Warmup
    warmup_steps=5 if TEST_MODE else 30,

    # Entraînement en float16
    fp16=True,

    # Affichage de la loss
    logging_steps=5 if TEST_MODE else 10,

    # Sauvegarde automatique
    save_strategy="steps",

    # Checkpoint toutes les 10 étapes pour le test
    save_steps=10 if TEST_MODE else 50,

    # Garder les deux derniers checkpoints
    save_total_limit=2,

    # Désactiver WandB
    report_to="none",

    # Conserver les colonnes nécessaires
    remove_unused_columns=False,

    # Chargement des données
    dataloader_num_workers=0,

    # Reproductibilité
    seed=SEED,

    # Optimiseur
    optim="adamw_torch",
)


# ============================================================
# 6. CRÉATION DU TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=args,

    train_dataset=PoemDS(train_ds),

    data_collator=collate
)


# ============================================================
# 7. GESTION DES CHECKPOINTS
# ============================================================

# Pendant le test, toujours démarrer sans ancien checkpoint.
# Pendant l'entraînement complet, reprendre un checkpoint
# du dossier FULL s'il existe.

if TEST_MODE:

    last = None
    print("\nTest : départ sans reprise de checkpoint.")

else:

    last = get_last_checkpoint(RUN_DIR)

    print(
        "\nReprise depuis :",
        last if last else "aucun checkpoint"
    )


# ============================================================
# 8. LANCEMENT DE L'ENTRAÎNEMENT
# ============================================================

print("\n========== DÉBUT ENTRAÎNEMENT ==========")

t0 = time.time()

res = trainer.train(
    resume_from_checkpoint=last
)

duration = time.time() - t0


# ============================================================
# 9. RÉSULTATS DE L'ENTRAÎNEMENT
# ============================================================

steps_done = res.global_step

sec_step = (
    res.metrics.get("train_runtime", duration)
    / max(steps_done, 1)
)

print("\n========== RÉSULTATS ==========")

print("Étapes terminées :", steps_done)

print(
    "Durée totale :",
    round(duration / 60, 2),
    "minutes"
)

print(
    "Temps moyen par étape :",
    round(sec_step, 2),
    "secondes"
)

print(
    "Train Loss :",
    res.metrics.get("train_loss")
)


# ============================================================
# 10. SAUVEGARDE DE L'ADAPTATEUR LORA
# ============================================================

model.save_pretrained(ADAPTER_DIR)

tok.save_pretrained(ADAPTER_DIR)

print("\nAdaptateur LoRA sauvegardé dans :")
print(ADAPTER_DIR)


# ============================================================
# 11. ESTIMATION DE L'ENTRAÎNEMENT COMPLET
# ============================================================

if TEST_MODE:

    estimated_minutes = (
        steps_full * sec_step / 60
    )

    print("\n========== ESTIMATION ==========")

    print(
        "Étapes complètes estimées :",
        steps_full
    )

    print(
        "Durée estimée :",
        round(estimated_minutes, 1),
        "minutes"
    )

    print("\nTest terminé !")

else:

    print("\nEntraînement complet terminé !")

GPU : Tesla T4
Exemples entraînement : 1962
Exemples validation : 255
Exemples test : 264

Mode : test
Checkpoints : /content/drive/MyDrive/projet_poesie/lora_runs/test
Adaptateur : /content/drive/MyDrive/projet_poesie/outputs/adapter_lora_test

Étapes estimées pour l'entraînement complet : 123
TEST RAPIDE : 20 étapes

Test : départ sans reprise de checkpoint.

========== DÉBUT ENTRAÎNEMENT ==========


Step,Training Loss
5,3.706170
10,3.546364
15,3.554578
20,3.493609



========== RÉSULTATS ==========
Étapes terminées : 20
Durée totale : 1.31 minutes
Temps moyen par étape : 3.91 secondes
Train Loss : 3.575180244445801

Adaptateur LoRA sauvegardé dans :
/content/drive/MyDrive/projet_poesie/outputs/adapter_lora_test

========== ESTIMATION ==========
Étapes complètes estimées : 123
Durée estimée : 8.0 minutes

Test terminé !


## 8. Évaluer le modèle

**En clair :** on mesure la qualité sur des poèmes que le modèle n'a **jamais vus** (jeu de test).

- **Perte par token** : L = NLL_totale / N_tokens
- **Perplexité par token** : PPL_token = exp(L)
- **Perplexité par mot** : PPL_mot = exp(NLL_totale / N_mots)

Plus c'est bas, mieux c'est. Les tokenizers de Qwen et de GPT-2 découpent le texte différemment : pour comparer deux modèles de familles différentes, utilisez la **perplexité par mot**, pas celle par token.

In [11]:
# Vérification de val_ds et des fonctions utiles
print("val_ds dispo :", "val_ds" in globals())
if "val_ds" in globals():
    print("Nombre de poèmes :", len(val_ds))
    print("Tokenizer dispo (tok) :", "tok" in globals())
    print("collate dispo :", "collate" in globals())

val_ds dispo : True
Nombre de poèmes : 255
Tokenizer dispo (tok) : True
collate dispo : True


In [12]:
# ====================================================
# ÉVALUATION QWEN + LORA
# ====================================================

import os
import math
import torch
import pandas as pd
import torch.nn.functional as F

@torch.no_grad()
def evaluate_rows(items, max_poems=40, bs=1):

    # Passer en mode évaluation
    model.eval()

    # Limiter le nombre de poèmes
    items = items[:max_poems]

    tot_nll = 0.0
    tot_tok = 0

    # Parcourir les exemples
    for i in range(0, len(items), bs):

        batch = items[i:i + bs]

        b = {
            k: v.to("cuda")
            for k, v in collate(batch).items()
        }

        # Prédictions du modèle
        logits = model(
            input_ids=b["input_ids"],
            attention_mask=b["attention_mask"]
        ).logits

        # Décalage pour prédire le token suivant
        sl = logits[:, :-1].float()
        lab = b["labels"][:, 1:]

        # Calcul de la loss
        nll = F.cross_entropy(
            sl.reshape(-1, sl.size(-1)),
            lab.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        tot_nll += nll.item()
        tot_tok += (lab != -100).sum().item()

        del logits, sl, lab, nll

    if tot_tok == 0:
        raise ValueError("Aucun token valide pour l'évaluation.")

    # Loss moyenne
    loss = tot_nll / tot_tok

    # Perplexité
    ppl = math.exp(loss)

    return {
        "poèmes": len(items),
        "loss/token": loss,
        "PPL token": ppl
    }


# ====================================================
# ÉVALUATION SUR VALIDATION
# ====================================================

n_eval = 40 if TEST_MODE else len(val_ds)

val_res = evaluate_rows(
    val_ds,
    max_poems=n_eval,
    bs=1
)

print("\n========== VALIDATION ==========")

for k, v in val_res.items():
    print(k, ":", round(v, 4) if isinstance(v, float) else v)


# ====================================================
# SAUVEGARDE DES RÉSULTATS
# ====================================================

os.makedirs(OUT_DIR, exist_ok=True)

result_path = os.path.join(
    OUT_DIR,
    f"eval_lora_{'test' if TEST_MODE else 'full'}.csv"
)

pd.DataFrame([
    {"jeu": "validation", **val_res}
]).to_csv(result_path, index=False)

print("\nRésultats sauvegardés :", result_path)
print("Évaluation terminée !")


========== VALIDATION ==========
poèmes : 40
loss/token : 3.5867
PPL token : 36.116

Résultats sauvegardés : /content/drive/MyDrive/projet_poesie/outputs/eval_lora_test.csv
Évaluation terminée !


## 9. Générer des poèmes

**En clair :** on donne au modèle un **thème** et une **première ligne**, et il écrit la suite. Réglages utilisés :

- `temperature = 0.8` : un peu de créativité (plus bas = plus sûr, plus haut = plus original mais moins cohérent) ;
- `top_p = 0.9` : on ne tire que parmi les mots les plus probables ;
- `repetition_penalty = 1.1` : évite de se répéter.

Le poème est limité à `stanzas` strophes, et un dernier vers coupé en cours de route est supprimé.

In [13]:
EOS_ID = tok.eos_token_id


@torch.no_grad()
def generate_poem(theme, first_line, stanzas=2, temperature=0.8, top_p=0.9, max_new_tokens=160, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    model.config.use_cache = True
    text = prompt_text(theme, first_line) + first_line.strip() + "\n"     # le poème commence par la 1re ligne
    inputs = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, do_sample=True, temperature=temperature, top_p=top_p,
                         repetition_penalty=1.1, max_new_tokens=max_new_tokens,
                         eos_token_id=EOS_ID, pad_token_id=tok.pad_token_id)
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    finished = EOS_ID in new_ids.tolist()
    gen = tok.decode(new_ids, skip_special_tokens=True).replace("\ufffd", "")
    poem = (first_line.strip() + "\n" + gen).strip()
    if not finished:                                    # génération coupée : on retire le vers incomplet
        lines = poem.split("\n")
        if len(lines) > 1 and not re.search(r"[.,;:!?\"')\-]$", lines[-1].strip()):
            poem = "\n".join(lines[:-1])
    strophes = [s for s in poem.split("\n\n") if s.strip()]
    return "\n\n".join(strophes[:stanzas]).strip()


print(generate_poem("the sea and memory", "The moon is silent over the sea", seed=1))

The moon is silent over the sea
A lullaby of silver light,
Whispers secrets through the night,
In waves that gently kiss.

Memories swell like water's flow,
Across the surface, deep and wide—
Of days we once danced on shore,
And nights where stars were so bright.


## 10. Les 10 poèmes de comparaison

**En clair :** mêmes débuts de poèmes que dans les tests précédents (DistilGPT-2), pour comparer. Les poèmes sont enregistrés dans Drive (`outputs/samples_lora.md`), avec un fichier CSV prêt à remplir pour **noter** les poèmes de 1 à 5 (cohérence, créativité, respect du style).

In [14]:
PROMPTS = [
    ("the sea and memory",  "The moon is silent over the sea"),
    ("autumn and loss",     "Autumn leaves fall on the silent road"),
    ("hope and dawn",       "Beyond the river waits the dawn"),
    ("the ocean",           "I heard the ocean in a shell"),
    ("loneliness",          "A bird returns to an empty garden"),
    ("night and names",     "The night remembers every name"),
    ("nature and voice",    "Your voice remains among the trees"),
    ("rain and the city",   "The rain has washed the city clean"),
    ("stars and solitude",  "Under the stars we walked alone"),
    ("winter and touch",    "The winter light touches my hands"),
]

md_out, eval_rows = [], []
for i, (theme, first) in enumerate(PROMPTS, 1):
    poem = generate_poem(theme, first, stanzas=3, seed=SEED + i)
    print(f"--- Poème {i} | thème : {theme} ---\n{poem}\n")
    md_out.append(f"## Poème {i}\n\nThème : {theme}\n\n```text\n{poem}\n```\n")
    eval_rows.append({"poeme_id": i, "theme": theme, "coherence": "", "creativite": "", "respect_style": "", "commentaire": ""})

suffix = "test" if TEST_MODE else "full"
with open(f"{OUT_DIR}/samples_lora_{suffix}.md", "w", encoding="utf-8") as f:
    f.write("# Poèmes générés (Qwen2.5-1.5B + LoRA)\n\n" + "\n".join(md_out))
pd.DataFrame(eval_rows).to_csv(f"{OUT_DIR}/human_eval_lora_{suffix}.csv", index=False)
print("Enregistré dans :", OUT_DIR)

--- Poème 1 | thème : the sea and memory ---
The moon is silent over the sea
Where secrets hold their breath in deep blue light,
A mirror of time's vast, unspoken tales.

In waves that dance, old memories weep,
Echoes of dreams untold or lost.
Each ripple whispers where we've been,
And what paths our lives have taken.

Whispers to the night, as stars align,
Memories drift like ghostly shadows,
Guiding us from shore to endless tide,
Through oceans wide, through skies so high.

--- Poème 2 | thème : autumn and loss ---
Autumn leaves fall on the silent road
A symphony of colors, soft and cold
The earth beneath them whispers tales
Of love lost in time's vast tomb

They drift like dreams across the sky,
Each one a memory, each one a cry
For every leaf that once was green
Now lies still, a fragment of their prime

In autumn's crisp embrace they dance
As if to say goodbye or come again
Yet here they lie, lifeless and pale
A mournful reminder of what can't be saved

--- Poème 3 | thème : hope 

In [15]:
# ============================================================
# ENTRAÎNEMENT COMPLET QWEN2.5 + LORA
# GPU : Tesla T4
# Époques : 1
# Validation + Checkpoints + Sauvegarde finale
# ============================================================

import os
import gc
import math
import time
import torch
import pandas as pd

from datetime import datetime
from transformers import (
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

# ============================================================
# 1. VÉRIFICATION DU GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU non disponible. Active le GPU T4 dans Colab."
    )

print("GPU utilisé :", torch.cuda.get_device_name(0))

# Vérifier les données préparées
assert len(train_ds) > 0, "Dataset train vide"
assert len(val_ds) > 0, "Dataset validation vide"

print("Exemples entraînement :", len(train_ds))
print("Exemples validation :", len(val_ds))


# ============================================================
# 2. PARAMÈTRES DE L'ENTRAÎNEMENT COMPLET
# ============================================================

TEST_MODE = False

EPOCHS = 1
BATCH = 1
ACCUM = 16

LR = 2e-4
SEED = 42

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Dossiers Google Drive
BASE = "/content/drive/MyDrive/projet_poesie"

OUT_DIR = f"{BASE}/outputs"

# Nouveau dossier pour éviter d'écraser le test précédent
RUN_DIR_FULL = (
    f"{BASE}/lora_runs/full_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)

ADAPTER_DIR_FULL = f"{RUN_DIR_FULL}/adapter_final"

os.makedirs(RUN_DIR_FULL, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print("\n========== CONFIGURATION ==========")
print("Mode : ENTRAÎNEMENT COMPLET")
print("Époques :", EPOCHS)
print("Batch size :", BATCH)
print("Gradient accumulation :", ACCUM)
print("Learning rate :", LR)
print("Dossier :", RUN_DIR_FULL)


# ============================================================
# 3. LIBÉRER LA MÉMOIRE DE L'ANCIEN MODÈLE
# ============================================================

# Le modèle du test de 20 étapes n'est plus nécessaire
# en mémoire, car son adaptateur a déjà été sauvegardé.

if "trainer" in globals():
    del trainer

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

print("\nMémoire GPU nettoyée.")


# ============================================================
# 4. RECHARGER QWEN DE BASE
# ============================================================

print("\nChargement du modèle Qwen de base...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16
).to("cuda")

# Désactiver le cache pour l'entraînement
model.config.use_cache = False

# Optimisation de la mémoire
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

model.enable_input_require_grads()


# ============================================================
# 5. CONFIGURER UN NOUVEL ADAPTATEUR LORA
# ============================================================

lora_cfg_full = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_cfg_full)

print("\n========== PARAMÈTRES LORA ==========")

model.print_trainable_parameters()


# ============================================================
# 6. DATASET PYTORCH
# ============================================================

class PoemDS(torch.utils.data.Dataset):

    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


train_dataset = PoemDS(train_ds)
val_dataset = PoemDS(val_ds)


# ============================================================
# 7. CALCULER LES ÉTAPES
# ============================================================

steps_full = math.ceil(
    len(train_dataset) / (BATCH * ACCUM)
) * EPOCHS

print("\nÉtapes estimées :", steps_full)


# ============================================================
# 8. CONFIGURATION TRAINING ARGUMENTS
# ============================================================

args_full = TrainingArguments(

    output_dir=RUN_DIR_FULL,

    # Une époque complète
    num_train_epochs=EPOCHS,

    # Ne pas limiter à 20 étapes
    max_steps=-1,

    # Batch
    per_device_train_batch_size=BATCH,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=ACCUM,

    # Optimisation
    learning_rate=LR,

    lr_scheduler_type="cosine",

    warmup_steps=10,

    fp16=True,

    # Affichage des résultats
    logging_steps=10,

    logging_strategy="steps",

    # Validation en fin d'époque
    eval_strategy="epoch",

    # Checkpoints
    save_strategy="steps",

    save_steps=25,

    save_total_limit=2,

    # Paramètres complémentaires
    report_to="none",

    remove_unused_columns=False,

    dataloader_num_workers=0,

    seed=SEED
)


# ============================================================
# 9. INITIALISER LE TRAINER
# ============================================================

trainer_full = Trainer(

    model=model,

    args=args_full,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=collate
)


# ============================================================
# 10. LANCER L'ENTRAÎNEMENT COMPLET
# ============================================================

print("\n========== DÉBUT ENTRAÎNEMENT ==========")

t0 = time.time()

# Nouveau départ : aucun checkpoint du test n'est repris
train_result = trainer_full.train(
    resume_from_checkpoint=None
)

duration = time.time() - t0

print("\n========== ENTRAÎNEMENT TERMINÉ ==========")

print("Étapes terminées :", train_result.global_step)

print("Durée totale :", round(duration / 60, 2), "minutes")

print(
    "Training Loss :",
    train_result.metrics.get("train_loss")
)


# ============================================================
# 11. ÉVALUATION SUR VALIDATION
# ============================================================

print("\n========== VALIDATION ==========")

val_metrics = trainer_full.evaluate()

val_loss = val_metrics["eval_loss"]

val_ppl = math.exp(val_loss)

print("Validation Loss :", round(val_loss, 4))
print("Validation Perplexity :", round(val_ppl, 4))


# ============================================================
# 12. SAUVEGARDER L'ADAPTATEUR LORA FINAL
# ============================================================

trainer_full.save_model(ADAPTER_DIR_FULL)

tok.save_pretrained(ADAPTER_DIR_FULL)

print("\nAdaptateur LoRA sauvegardé :")
print(ADAPTER_DIR_FULL)


# ============================================================
# 13. SAUVEGARDER LES RÉSULTATS
# ============================================================

results_df = pd.DataFrame([{

    "model": MODEL_NAME,

    "epochs": EPOCHS,

    "steps": train_result.global_step,

    "train_loss": train_result.metrics.get("train_loss"),

    "val_loss": val_loss,

    "val_ppl": val_ppl,

    "duration_minutes": duration / 60

}])

results_path = f"{RUN_DIR_FULL}/training_results.csv"

results_df.to_csv(
    results_path,
    index=False
)

print("\nRésultats enregistrés :", results_path)

print("\nENTRAÎNEMENT COMPLET TERMINÉ !")

GPU utilisé : Tesla T4
Exemples entraînement : 1962
Exemples validation : 255

========== CONFIGURATION ==========
Mode : ENTRAÎNEMENT COMPLET
Époques : 1
Batch size : 1
Gradient accumulation : 16
Learning rate : 0.0002
Dossier : /content/drive/MyDrive/projet_poesie/lora_runs/full_20261021_220450

Mémoire GPU nettoyée.

Chargement du modèle Qwen de base...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


========== PARAMÈTRES LORA ==========
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

Étapes estimées : 123

========== DÉBUT ENTRAÎNEMENT ==========


Epoch,Training Loss,Validation Loss
1,3.470007,3.426715



========== ENTRAÎNEMENT TERMINÉ ==========
Étapes terminées : 123
Durée totale : 16.42 minutes
Training Loss : 3.4391755670066773

========== VALIDATION ==========


Training Loss,Validation Loss,Epoch
3.470007,3.426715,1


Validation Loss : 3.4267
Validation Perplexity : 30.7754

Adaptateur LoRA sauvegardé :
/content/drive/MyDrive/projet_poesie/lora_runs/full_20261021_220450/adapter_final

Résultats enregistrés : /content/drive/MyDrive/projet_poesie/lora_runs/full_20261021_220450/training_results.csv

ENTRAÎNEMENT COMPLET TERMINÉ !


In [22]:
# ============================================================
# GÉNÉRATION DE POÈMES AVEC QWEN + LORA FINAL
# Comparaison avec le test de 20 étapes
# ============================================================

import os
import re
import torch
import pandas as pd

# 1. Vérifier que le modèle complet est disponible
if "trainer_full" not in globals():
    raise RuntimeError(
        "Le modèle complet n'est plus en mémoire. "
        "Il faudra recharger l'adaptateur sauvegardé."
    )

model_final = trainer_full.model
model_final.eval()
model_final.config.use_cache = True

# 2. Définir les mêmes thèmes que pour le test précédent
PROMPTS = [
    ("the sea and memory", "The moon is silent over the sea"),
    ("autumn and loss", "Autumn leaves fall on the silent road"),
    ("hope and dawn", "Beyond the river waits the dawn"),
    ("the ocean", "I heard the ocean in a shell"),
    ("loneliness", "A bird returns to an empty garden"),
    ("night and names", "The night remembers every name"),
    ("nature and voice", "Your voice remains among the trees"),
    ("rain and the city", "The rain has washed the city clean"),
    ("stars and solitude", "Under the stars we walked alone"),
    ("winter and touch", "The winter light touches my hands")
]

# 3. Fonction de génération
@torch.no_grad()
def generate_final(theme, first_line, seed=42):

    torch.manual_seed(seed)

    prompt = (
        prompt_text(theme, first_line)
        + first_line.strip()
        + "\n"
    )

    inputs = tok(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cuda")

    outputs = model_final.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.1,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id
    )

    # Extraire uniquement les nouveaux tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    generated = tok.decode(
        new_tokens,
        skip_special_tokens=True
    )

    poem = first_line.strip() + "\n" + generated.strip()

    return poem.strip()


# 4. Générer les 10 poèmes
resultats = []

for i, (theme, first) in enumerate(PROMPTS, 1):

    poem = generate_final(
        theme,
        first,
        seed=SEED + i
    )

    print(f"\n========== POÈME {i} ==========")
    print("Thème :", theme)
    print(poem)

    resultats.append({
        "poeme_id": i,
        "theme": theme,
        "premiere_ligne": first,
        "poeme": poem
    })


# 5. Sauvegarder les résultats sur Drive
os.makedirs(OUT_DIR, exist_ok=True)

fichier = os.path.join(
    OUT_DIR,
    "poemes_qwen_lora_full.csv"
)

pd.DataFrame(resultats).to_csv(
    fichier,
    index=False,
    encoding="utf-8-sig"
)

print("\n=================================")
print("10 poèmes générés et sauvegardés !")
print("Fichier :", fichier)


========== POÈME 1 ==========
Thème : the sea and memory
The moon is silent over the sea
and the wind blows on its surface;
the waves, as if it were not there,
rise and fall in silence.
They say that the moon has no voice
to speak of what it sees; yet here
I hear her speaking clearly to me
and I have learned from her how to love.
In my heart she sings softly:
She speaks of something that happened
that is far beyond our reach
but for which we can hope
for it will come again.
My heart feels glad at her words,
which give me the strength to bear
all that is hard and painful.
The moon is still silent on the sea,
but her voice has made me strong.

========== POÈME 2 ==========
Thème : autumn and loss
Autumn leaves fall on the silent road
and in my heart I feel the same
as the old trees that stand so still
in their silence, their eyes closed
to the world outside.
I'm lost among these quiet shadows,
alone in them as they're alone;
the only thing to say is "goodbye"
to what we have been, what'

In [23]:
# ===================================================
# ÉVALUATION FINALE : QWEN + LORA COMPLET
# 255 poèmes de validation
# ===================================================

import math
import pandas as pd
import torch

# Récupérer le modèle entraîné
model_final = trainer_full.model

# Mode évaluation
model_final.eval()
model_final.config.use_cache = False

# Évaluer sur tous les exemples de validation
metrics = trainer_full.evaluate(
    eval_dataset=PoemDS(val_ds)
)

val_loss = metrics["eval_loss"]
val_ppl = math.exp(val_loss)

print("========== ÉVALUATION FINALE ==========")
print("Nombre de poèmes :", len(val_ds))
print("Validation Loss :", round(val_loss, 4))
print("Validation Perplexity :", round(val_ppl, 4))

# Enregistrer les résultats
resultats = pd.DataFrame([{
    "modele": "Qwen2.5 + LoRA complet",
    "etapes": 123,
    "poemes_validation": len(val_ds),
    "validation_loss": val_loss,
    "validation_ppl": val_ppl
}])

fichier = f"{OUT_DIR}/evaluation_finale_qwen_lora.csv"

resultats.to_csv(fichier, index=False)

print("\nRésultats sauvegardés :", fichier)

Training Loss,Validation Loss,Epoch
3.470007,3.426715,1


========== ÉVALUATION FINALE ==========
Nombre de poèmes : 255
Validation Loss : 3.4267
Validation Perplexity : 30.7754

Résultats sauvegardés : /content/drive/MyDrive/projet_poesie/outputs/evaluation_finale_qwen_lora.csv


# **evaluation**

In [16]:
# =====================================================
# ÉTAPE 1 : VÉRIFICATION AVANT L'ÉVALUATION LORA
# =====================================================

import os
import torch

BASE = "/content/drive/MyDrive/projet_poesie"
ADAPTER_TEST = f"{BASE}/outputs/adapter_lora_test"

print("========== VÉRIFICATIONS ==========")

# 1. Vérifier le GPU
print("GPU disponible :", torch.cuda.is_available())

# 2. Vérifier l'adaptateur LoRA test (20 étapes)
print("Adaptateur LoRA test :", os.path.isdir(ADAPTER_TEST))

# 3. Vérifier les données de validation
print("val_ds disponible :", "val_ds" in globals())

if "val_ds" in globals():
    print("Nombre de poèmes :", len(val_ds))

# 4. Vérifier les fonctions nécessaires
print("Fonction collate :", callable(globals().get("collate")))

print("Fonction evaluate_rows :", callable(globals().get("evaluate_rows")))

# 5. Vérifier le modèle de base
print("MODEL_NAME disponible :", "MODEL_NAME" in globals())

if "MODEL_NAME" in globals():
    print("Modèle :", MODEL_NAME)

========== VÉRIFICATIONS ==========
GPU disponible : True
Adaptateur LoRA test : True
val_ds disponible : True
Nombre de poèmes : 255
Fonction collate : True
Fonction evaluate_rows : True
MODEL_NAME disponible : True
Modèle : Qwen/Qwen2.5-1.5B-Instruct


In [17]:
# =====================================================
# ÉTAPE 2 : VÉRIFICATION DES DONNÉES ENCODÉES
# =====================================================

import torch

print("========== STRUCTURE DES DONNÉES ==========")

# Vérifier le type des données
print("Type de val_ds :", type(val_ds))

# Vérifier le nombre de poèmes
print("Nombre de poèmes :", len(val_ds))

# Examiner le premier poème encodé
exemple = val_ds[0]

print("\nType du premier exemple :", type(exemple))

if isinstance(exemple, dict):
    print("\nColonnes disponibles :", list(exemple.keys()))

    for cle, valeur in exemple.items():
        if isinstance(valeur, (list, tuple, torch.Tensor)):
            print(cle, ": longueur =", len(valeur))
        else:
            print(cle, ": type =", type(valeur))

# Vérifier le tokenizer
print("\nTokenizer disponible :",
      "tokenizer" in globals())

========== STRUCTURE DES DONNÉES ==========
Type de val_ds : <class 'list'>
Nombre de poèmes : 255

Type du premier exemple : <class 'dict'>

Colonnes disponibles : ['input_ids', 'labels', 'n_words']
input_ids : longueur = 225
labels : longueur = 225
n_words : type = <class 'int'>

Tokenizer disponible : False


In [18]:
# =====================================================
# ÉTAPE 3 : RESTAURER TOKENIZER ET COLLATE
# =====================================================

import torch
from transformers import AutoTokenizer

# 1. Charger le tokenizer du modèle Qwen
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# Définir le token de remplissage si nécessaire
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer chargé avec succès !")

# 2. Restaurer la fonction collate
def collate(exemples):

    input_ids = [
        torch.as_tensor(ex["input_ids"], dtype=torch.long)
        for ex in exemples
    ]

    labels = [
        torch.as_tensor(ex["labels"], dtype=torch.long)
        for ex in exemples
    ]

    # Remplir les séquences à la même longueur
    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    labels = torch.nn.utils.rnn.pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    # Créer le masque d'attention
    attention_mask = torch.zeros_like(input_ids)

    for i, ex in enumerate(exemples):
        attention_mask[i, :len(ex["input_ids"])] = 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# 3. Vérifier le fonctionnement
batch_test = collate([val_ds[0]])

print("\n========== VÉRIFICATION ==========")
print("Tokenizer disponible :", tokenizer is not None)
print("Fonction collate disponible :", callable(collate))
print("Shape input_ids :", batch_test["input_ids"].shape)
print("Shape labels :", batch_test["labels"].shape)
print("Shape attention_mask :", batch_test["attention_mask"].shape)

print("\nÉtape 3 terminée !")

Tokenizer chargé avec succès !

========== VÉRIFICATION ==========
Tokenizer disponible : True
Fonction collate disponible : True
Shape input_ids : torch.Size([1, 225])
Shape labels : torch.Size([1, 225])
Shape attention_mask : torch.Size([1, 225])

Étape 3 terminée !


In [19]:
# =====================================================
# ÉTAPE 4 : FONCTION D'ÉVALUATION QWEN + LORA
# =====================================================

import math
import torch
import torch.nn.functional as F

@torch.no_grad()
def evaluer_lora(modele, donnees):

    modele.eval()

    total_loss = 0.0
    total_tokens = 0

    device = next(modele.parameters()).device

    for i, exemple in enumerate(donnees):

        # Préparer le poème
        batch = collate([exemple])

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        # Prédictions du modèle
        outputs = modele(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        logits = outputs.logits[:, :-1, :].float()
        labels = batch["labels"][:, 1:]

        # Calcul de la loss
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        total_loss += loss.item()
        total_tokens += (labels != -100).sum().item()

        # Afficher la progression
        if (i + 1) % 50 == 0:
            print(f"Progression : {i + 1}/{len(donnees)}")

        del outputs, logits, labels, loss

    if total_tokens == 0:
        raise ValueError("Aucun token valide pour l'évaluation.")

    # Loss moyenne par token
    validation_loss = total_loss / total_tokens

    # Perplexité
    validation_ppl = math.exp(validation_loss)

    return validation_loss, validation_ppl


# =====================================================
# VÉRIFICATION
# =====================================================

print("========== ÉTAPE 4 ==========")

print(
    "Fonction d'évaluation disponible :",
    callable(evaluer_lora)
)

print("Étape 4 terminée !")

========== ÉTAPE 4 ==========
Fonction d'évaluation disponible : True
Étape 4 terminée !


In [21]:
# =====================================================
# ÉTAPE 5 : CHARGEMENT DU LORA TEST (20 ÉTAPES)
# =====================================================

import os
import gc
import torch

from transformers import AutoModelForCausalLM
from peft import PeftModel

# 1. Paramètres
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

ADAPTER_TEST = (
    "/content/drive/MyDrive/projet_poesie/"
    "outputs/adapter_lora_test"
)

# 2. Vérifications
assert torch.cuda.is_available(), "GPU indisponible."
assert os.path.isdir(ADAPTER_TEST), "Adaptateur LoRA introuvable."
assert "val_ds" in globals(), "Données de validation absentes."
assert len(val_ds) == 255, "Nombre de poèmes incorrect."

print("GPU disponible :", torch.cuda.is_available())
print("Poèmes de validation :", len(val_ds))

# 3. Libérer la mémoire GPU
for nom in [
    "model",
    "model_test",
    "base_model",
    "model_final",
    "trainer",
    "trainer_full"
]:
    if nom in globals():
        del globals()[nom]

gc.collect()
torch.cuda.empty_cache()

# 4. Charger Qwen2.5
print("\nChargement du modèle Qwen2.5...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

# 5. Charger l'adaptateur LoRA test
print("\nChargement de l'adaptateur LoRA test...")

model_test = PeftModel.from_pretrained(
    base_model,
    ADAPTER_TEST
)

# 6. Activer le mode évaluation
model_test.eval()
model_test.config.use_cache = False

# 7. Vérification finale
print("\n========== ÉTAPE 5 ==========")
print("Modèle :", MODEL_NAME)
print("Modèle chargé :", model_test is not None)
print("Mode évaluation :", not model_test.training)
print("Poèmes disponibles :", len(val_ds))

print("\nÉtape 5 terminée !")

GPU disponible : True
Poèmes de validation : 255

Chargement du modèle Qwen2.5...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Chargement de l'adaptateur LoRA test...

========== ÉTAPE 5 ==========
Modèle : Qwen/Qwen2.5-1.5B-Instruct
Modèle chargé : True
Mode évaluation : True
Poèmes disponibles : 255

Étape 5 terminée !


In [22]:
# =====================================================
# ÉTAPE 6 : ÉVALUATION DU LORA TEST (20 ÉTAPES)
# =====================================================

import os
import csv
import math
import torch
import torch.nn.functional as F

# 1. Vérifications
assert "model_test" in globals(), "Modèle LoRA test absent."
assert "val_ds" in globals(), "Données de validation absentes."
assert "collate" in globals(), "Fonction collate absente."

print("========================================")
print("ÉVALUATION DU MODÈLE QWEN2.5 + LORA TEST")
print("========================================")

print("Nombre de poèmes :", len(val_ds))

# 2. Mettre le modèle en mode évaluation
model_test.eval()

total_loss = 0.0
total_tokens = 0

device = next(model_test.parameters()).device

# 3. Évaluer les 255 poèmes
with torch.inference_mode():

    for i, exemple in enumerate(val_ds):

        batch = collate([exemple])

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        outputs = model_test(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        logits = outputs.logits[:, :-1, :].float()
        labels = batch["labels"][:, 1:]

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        total_loss += loss.item()
        total_tokens += (labels != -100).sum().item()

        if (i + 1) % 50 == 0:
            print(f"Progression : {i + 1}/{len(val_ds)}")

        del outputs, logits, labels, loss

# 4. Calculer les résultats
assert total_tokens > 0, "Aucun token valide."

validation_loss = total_loss / total_tokens
validation_ppl = math.exp(validation_loss)

# 5. Afficher les résultats
print("\n========================================")
print("RÉSULTATS FINAUX DU LORA TEST")
print("========================================")

print("Modèle : Qwen2.5 + LoRA test")
print("Étapes d'entraînement : 20")
print("Poèmes évalués :", len(val_ds))
print(f"Validation Loss : {validation_loss:.6f}")
print(f"Validation PPL : {validation_ppl:.4f}")

# 6. Sauvegarder les résultats dans Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/projet_poesie/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

fichier = os.path.join(
    OUTPUT_DIR,
    "evaluation_lora_test_255.csv"
)

with open(fichier, "w", newline="", encoding="utf-8") as f:

    writer = csv.writer(f)

    writer.writerow([
        "modele",
        "etapes",
        "poemes_validation",
        "validation_loss",
        "validation_ppl"
    ])

    writer.writerow([
        "Qwen2.5 + LoRA test",
        20,
        len(val_ds),
        validation_loss,
        validation_ppl
    ])

print("\nRésultats sauvegardés :", fichier)
print("\n✅ ÉTAPE 6 TERMINÉE !")

ÉVALUATION DU MODÈLE QWEN2.5 + LORA TEST
Nombre de poèmes : 255
Progression : 50/255
Progression : 100/255
Progression : 150/255
Progression : 200/255
Progression : 250/255

RÉSULTATS FINAUX DU LORA TEST
Modèle : Qwen2.5 + LoRA test
Étapes d'entraînement : 20
Poèmes évalués : 255
Validation Loss : 3.507707
Validation PPL : 33.3716

Résultats sauvegardés : /content/drive/MyDrive/projet_poesie/outputs/evaluation_lora_test_255.csv

✅ ÉTAPE 6 TERMINÉE !


# **Étape 7 — Génération de poèmes avec Qwen2.5 + LoRA final (123 étapes)**

In [23]:
# ============================================================
# ÉTAPE 7 : GÉNÉRATION DE POÈMES AVEC QWEN2.5 + LORA FINAL
# ============================================================

import os
import torch
from transformers import set_seed

# 1. Chemin de l'adaptateur final (123 étapes)
ADAPTER_FINAL = (
    "/content/drive/MyDrive/projet_poesie/"
    "lora_runs/full_20261021_220450/adapter_final"
)

# 2. Vérifications
assert os.path.isdir(ADAPTER_FINAL), "Adaptateur final introuvable."
assert "model_test" in globals(), "Le modèle Qwen + LoRA test est absent."
assert "tok" in globals(), "Tokenizer tok indisponible."

print("Adaptateur final trouvé !")

# 3. Charger l'adaptateur final dans le modèle existant
if "final" not in model_test.peft_config:
    model_test.load_adapter(
        ADAPTER_FINAL,
        adapter_name="final"
    )

model_test.set_adapter("final")
model_test.eval()

print("LoRA final activé avec succès !")

# 4. Fonction de génération
def generer_poeme(theme, premiere_ligne):

    system = (
        "You are a poet. Write original English poems "
        "with clear imagery and a coherent theme."
    )

    user = (
        f"Write a poem about: {theme}.\n"
        f"Start with this line: {premiere_ligne}"
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

    prompt = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tok(
        prompt,
        return_tensors="pt"
    )

    device = next(model_test.parameters()).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        outputs = model_test.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tok.eos_token_id
        )

    nouveaux_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    poeme = tok.decode(
        nouveaux_tokens,
        skip_special_tokens=True
    )

    return poeme.strip()


# 5. Thèmes à tester
tests = [
    ("Nature", "The wind whispers through the trees"),
    ("Hope", "Beyond the darkness comes the light"),
    ("Loneliness", "A lonely bird sings in the night")
]

# 6. Générer trois poèmes
set_seed(42)

print("\n========================================")
print("GÉNÉRATION AVEC QWEN2.5 + LORA FINAL")
print("========================================")

poemes_finaux = []

for i, (theme, premiere_ligne) in enumerate(tests, 1):

    poeme = generer_poeme(theme, premiere_ligne)

    poemes_finaux.append({
        "theme": theme,
        "premiere_ligne": premiere_ligne,
        "poeme": poeme
    })

    print(f"\n========== POÈME {i} ==========")
    print("Thème :", theme)
    print("Première ligne :", premiere_ligne)
    print("\n", poeme)
    print("=" * 40)

print("\n✅ ÉTAPE 7 TERMINÉE !")


✅ ÉTAPE 7 TERMINÉE !


In [25]:
# =====================================================
# VÉRIFICATION DES POÈMES GÉNÉRÉS À L'ÉTAPE 7
# =====================================================

if "poemes_finaux" in globals():

    print("Nombre de poèmes :", len(poemes_finaux))

    if len(poemes_finaux) > 0:

        for i, resultat in enumerate(poemes_finaux, 1):

            print(f"\n========== POÈME {i} ==========")
            print("Thème :", resultat["theme"])
            print("Première ligne :", resultat["premiere_ligne"])
            print("\nPoème généré :")
            print(resultat["poeme"])
            print("=" * 40)

    else:
        print("Aucun poème n'a été enregistré.")

else:
    print("La variable poemes_finaux est introuvable.")

print("\n✅ Vérification terminée !")

La variable poemes_finaux est introuvable.

✅ Vérification terminée !


In [32]:
# ============================================================
# ÉTAPE 7 : GÉNÉRATION AVEC QWEN2.5 + LORA FINAL
# ============================================================

import os
import torch

# 1. Vérifier le modèle
assert "model_test" in globals(), "Modèle Qwen absent."

# 2. Récupérer le tokenizer
if "tok" not in globals():
    if "tokenizer" in globals():
        tok = tokenizer
    else:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(
            "Qwen/Qwen2.5-1.5B-Instruct"
        )

# 3. Activer le LoRA final
ADAPTER_FINAL = (
    "/content/drive/MyDrive/projet_poesie/"
    "lora_runs/full_20261021_220450/adapter_final"
)

assert os.path.isdir(ADAPTER_FINAL), "Adaptateur final introuvable."

if "final" not in model_test.peft_config:
    model_test.load_adapter(
        ADAPTER_FINAL,
        adapter_name="final"
    )

model_test.set_adapter("final")
model_test.eval()

print("Modèle LoRA final activé !", flush=True)

# 4. Créer la fonction de génération
def generer_poeme(theme, premiere_ligne):

    messages = [
        {
            "role": "system",
            "content": "You are a poet. Write original English poems."
        },
        {
            "role": "user",
            "content": (
                f"Write a poem about {theme}. "
                f"Start with this line: {premiere_ligne}"
            )
        }
    ]

    prompt = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tok(prompt, return_tensors="pt")

    device = next(model_test.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model_test.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tok.eos_token_id
        )

    nouveaux_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tok.decode(
        nouveaux_tokens,
        skip_special_tokens=True
    ).strip()

# 5. Définir les trois thèmes
tests = [
    ("Nature", "The wind whispers through the trees"),
    ("Hope", "Beyond the darkness comes the light"),
    ("Loneliness", "A lonely bird sings in the night")
]

# 6. Générer les poèmes
poemes_finaux = []

for i, (theme, premiere_ligne) in enumerate(tests, 1):

    print(f"Génération du poème {i}/3...", flush=True)

    poeme = generer_poeme(theme, premiere_ligne)

    poemes_finaux.append({
        "theme": theme,
        "premiere_ligne": premiere_ligne,
        "poeme": poeme
    })

    print(f"\n========== POÈME {i} ==========")
    print("Thème :", theme)
    print("Poème :", poeme, flush=True)

# 7. Vérifier le résultat
print("\nNombre de poèmes :", len(poemes_finaux))
print("ÉTAPE 7 TERMINÉE !")

Modèle LoRA final activé !
Génération du poème 1/3...

========== POÈME 1 ==========
Thème : Nature
Poème : The wind whispers through the trees
and the earth gives way underfoot.
I am no longer afraid of death,
for it is nothing more than me-
a memory, a story, or just an idea.
Génération du poème 2/3...

========== POÈME 2 ==========
Thème : Hope
Poème : Beyond the darkness comes the light
When you walk through it, you will find the path
Where there is nothing to be found.
The only thing that can happen in the world,
Is when someone gives something away
To another person. The most beautiful gift
That has ever been given, was by an old man
Who gave his life for us all. And he said:
"Give me your hand and let's go."
Génération du poème 3/3...

========== POÈME 3 ==========
Thème : Loneliness
Poème : A lonely bird sings in the night
That can't sing at all, but only moans,
And flies away from its own nest-
It's never heard of before.
There was once an old man who had no children:
His wife

In [35]:
# =====================================================
# DIAGNOSTIC COMPLET - QWEN2.5 + LORA
# =====================================================

import IPython.display

# Vérifier les variables disponibles
modele_disponible = "model_test" in globals()
fonction_disponible = "generer_poeme" in globals()
poemes_disponibles = "poemes_finaux" in globals()

nombre_poemes = (
    len(globals()["poemes_finaux"])
    if poemes_disponibles
    else 0
)

# Afficher tous les résultats ensemble
message = f"""
<h3>DIAGNOSTIC DU MODÈLE</h3>
<p>Modèle disponible : {modele_disponible}</p>
<p>Fonction de génération disponible : {fonction_disponible}</p>
<p>Liste des poèmes disponible : {poemes_disponibles}</p>
<p>Nombre de poèmes : {nombre_poemes}</p>
<h3>DIAGNOSTIC TERMINÉ</h3>
"""

IPython.display.display(IPython.display.HTML(message))

In [34]:
# =====================================================
# ÉTAPE 8 : SAUVEGARDE DES POÈMES GÉNÉRÉS
# QWEN2.5 + LORA FINAL (123 ÉTAPES)
# =====================================================

import os
import csv

# 1. Vérifier les poèmes générés
assert "poemes_finaux" in globals(), "Liste des poèmes introuvable."
assert len(poemes_finaux) == 3, "Il faut exactement 3 poèmes."

# 2. Définir le dossier de sauvegarde
OUTPUT_DIR = "/content/drive/MyDrive/projet_poesie/outputs"

assert os.path.isdir(
    "/content/drive/MyDrive"
), "Google Drive n'est pas monté."

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Sauvegarder les poèmes au format TXT
fichier_txt = os.path.join(
    OUTPUT_DIR,
    "poemes_qwen_lora_final.txt"
)

with open(fichier_txt, "w", encoding="utf-8") as f:

    f.write("POÈMES GÉNÉRÉS AVEC QWEN2.5 + LORA FINAL\n")
    f.write("Entraînement : 123 étapes\n")
    f.write("=" * 50 + "\n\n")

    for i, resultat in enumerate(poemes_finaux, 1):

        f.write(f"POÈME {i}\n")
        f.write(f"Thème : {resultat['theme']}\n")
        f.write(
            f"Première ligne : {resultat['premiere_ligne']}\n\n"
        )
        f.write(resultat["poeme"] + "\n\n")
        f.write("=" * 50 + "\n\n")

# 4. Sauvegarder les poèmes au format CSV
fichier_csv = os.path.join(
    OUTPUT_DIR,
    "poemes_qwen_lora_final.csv"
)

with open(
    fichier_csv,
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "modele",
        "etapes",
        "theme",
        "premiere_ligne",
        "poeme"
    ])

    for resultat in poemes_finaux:

        writer.writerow([
            "Qwen2.5 + LoRA final",
            123,
            resultat["theme"],
            resultat["premiere_ligne"],
            resultat["poeme"]
        ])

# 5. Vérification de la sauvegarde
print("========== SAUVEGARDE ==========")

print("Nombre de poèmes :", len(poemes_finaux))
print("Fichier TXT :", os.path.exists(fichier_txt))
print("Fichier CSV :", os.path.exists(fichier_csv))

print("\nPoèmes sauvegardés dans Google Drive !")
print("Dossier :", OUTPUT_DIR)

print("\n✅ ÉTAPE 8 TERMINÉE AVEC SUCCÈS !")


✅ ÉTAPE 8 TERMINÉE AVEC SUCCÈS !


In [37]:
# =====================================================
# SAUVEGARDE DES 3 NOUVEAUX POÈMES
# QWEN2.5 + LORA FINAL (123 ÉTAPES)
# =====================================================

import os
import csv

# 1. Vérifier les poèmes générés
assert "poemes_finaux" in globals(), "Liste des poèmes introuvable."
assert len(poemes_finaux) == 3, "Les trois poèmes sont introuvables."

# 2. Définir le dossier Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/projet_poesie/outputs"

assert os.path.isdir("/content/drive/MyDrive"), \
    "Google Drive n'est pas monté."

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Créer un fichier CSV avec un nom facile à retrouver
fichier_csv = os.path.join(
    OUTPUT_DIR,
    "MES_3_NOUVEAUX_POEMES_LORA_FINAL.csv"
)

with open(fichier_csv, "w", newline="", encoding="utf-8-sig") as f:

    writer = csv.writer(f)

    writer.writerow([
        "Theme",
        "Premiere ligne",
        "Poeme"
    ])

    for resultat in poemes_finaux:

        writer.writerow([
            resultat["theme"],
            resultat["premiere_ligne"],
            resultat["poeme"]
        ])

# 4. Vérifier le fichier sauvegardé
with open(fichier_csv, "r", encoding="utf-8-sig", newline="") as f:
    lignes = list(csv.DictReader(f))

assert len(lignes) == 3, "La sauvegarde est incomplète."

# 5. Afficher le résultat
print("========== SAUVEGARDE ==========")
print("Fichier sauvegardé :", os.path.isfile(fichier_csv))
print("Nombre de poèmes :", len(lignes))
print("Nom du fichier :", os.path.basename(fichier_csv))
print("Dossier :", OUTPUT_DIR)

print("\n✅ SAUVEGARDE TERMINÉE AVEC SUCCÈS !")

========== SAUVEGARDE ==========
Fichier sauvegardé : True
Nombre de poèmes : 3
Nom du fichier : MES_3_NOUVEAUX_POEMES_LORA_FINAL.csv
Dossier : /content/drive/MyDrive/projet_poesie/outputs

✅ SAUVEGARDE TERMINÉE AVEC SUCCÈS !
